In [1]:
import warnings

warnings.filterwarnings("ignore")

from core.backtesting import BacktestingEngine

backtesting = BacktestingEngine(load_cached_data=True)

In [4]:
from app.controllers.directional_trading.trend_example import TrendExampleControllerConfig
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
import datetime
from decimal import Decimal


# Controller configuration
connector_name = "binance_perpetual"
trading_pair = "1000BONK-USDT"
interval = "1m"
ema_short: int = 8
ema_medium: int = 29
ema_long: int = 100
total_amount_quote = 1000
max_executors_per_side = 2
time_limit = 60 * 60 * 12
cooldown_time = 60 * 15
take_profit = 0.04
stop_loss = 0.02
trailing_stop_activation_price = 0.015
trailing_stop_trailing_delta = 0.005


# Creating the instance of the configuration and the controller
config = TrendExampleControllerConfig(
    connector_name=connector_name,
    trading_pair=trading_pair,
    candles_connector=connector_name,
    candles_trading_pair=trading_pair,
    interval=interval,
    take_profit=Decimal(take_profit),
    stop_loss=Decimal(stop_loss),
    trailing_stop=TrailingStop(activation_price=Decimal(trailing_stop_activation_price), trailing_delta=Decimal(trailing_stop_trailing_delta)),
    total_amount_quote=Decimal(total_amount_quote),
    time_limit=time_limit,
    max_executors_per_side=max_executors_per_side,
    cooldown_time=cooldown_time,
    ema_short=ema_short,
    ema_medium=ema_medium,
    ema_long=ema_long,
)

In [5]:
# Running the backtesting this will output a backtesting result object that has built in methods to visualize the results

start = int(datetime.datetime(2024, 11, 1).timestamp())
end = int(datetime.datetime(2024, 11, 16).timestamp())

backtesting_result = await backtesting.run_backtesting(config, start, end, "1m")

In [6]:
import plotly.graph_objects as go

# Let's see what is inside the backtesting results
print(backtesting_result.get_results_summary())
fig = backtesting_result.get_backtesting_figure()
# Add EMAs
candles_df = backtesting_result.processed_data
ema_fast = f'EMA_{ema_short}'
ema_med = f'EMA_{ema_medium}'
ema_slow = f'EMA_{ema_long}'

fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_fast],
                         line=dict(color='#00FF00', width=2),
                         name='Fast EMA'))
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_med],
                         line=dict(color='#FFA500', width=2),
                         name='Medium EMA'))
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_slow],
                         line=dict(color='#0000FF', width=2),
                         name='Slow EMA'))



Net PNL: $138.64 (13.86%) | Max Drawdown: $-424.70 (-42.22%)
Total Volume ($): 699000.00 | Sharpe Ratio: 0.07 | Profit Factor: 1.04
Total Executors: 699 | Accuracy Long: 0.57 | Accuracy Short: 0.49
Close Types: Take Profit: 39 | Stop Loss: 323 | Time Limit: 3 |
             Trailing Stop: 334 | Early Stop: 0



In [ ]:
# 2. The executors dataframe: this is the dataframe that contains the information of the orders that were executed
import pandas as pd

executors_df = backtesting_result.executors_df
executors_df.head()

### Backtesting Analysis

### Scatter of PNL per Trade
This bar chart illustrates the PNL for each individual trade. Positive PNLs are shown in green and negative PNLs in red, providing a clear view of profitable vs. unprofitable trades.


In [ ]:
import plotly.express as px

# Create a new column for profitability
executors_df['profitable'] = executors_df['net_pnl_quote'] > 0

# Create the scatter plot
fig = px.scatter(
    executors_df,
    x="timestamp",
    y='net_pnl_quote',
    title='PNL per Trade',
    color='profitable',
    color_discrete_map={True: 'green', False: 'red'},
    labels={'timestamp': 'Timestamp', 'net_pnl_quote': 'Net PNL (Quote)'},
    hover_data=['filled_amount_quote', 'side']
)

# Customize the layout
fig.update_layout(
    xaxis_title="Timestamp",
    yaxis_title="Net PNL (Quote)",
    legend_title="Profitable",
    font=dict(size=12, color="white"),
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0.8)',  # Dark background
    paper_bgcolor='rgba(0,0,0,0.8)',  # Dark background for the entire plot area
    xaxis=dict(gridcolor="gray"),
    yaxis=dict(gridcolor="gray")
)

# Add a horizontal line at y=0 to clearly separate profits and losses
fig.add_hline(y=0, line_dash="dash", line_color="lightgray")

# Show the plot
fig.show()

### Histogram of PNL Distribution
The histogram displays the distribution of PNL values across all trades. It helps in understanding the frequency and range of profit and loss outcomes.


In [ ]:
fig = px.histogram(executors_df, x='net_pnl_quote', title='PNL Distribution')
fig.show()


# Conclusion
We can see that the indicator has potential to bring good signals to trade and might be interesting to see how we can design a market maker that shifts the mid price based on this indicator.
A lot of the short signals are wrong but if we zoom in into the loss signals we can see that the losses are not that big and the wins are bigger and if we had implemented the trailing stop feature probably a lot of them are going to be profits.

# Next steps
- Filter only the loss signals and understand what you can do to prevent them
- Try different configuration values for the indicator
- Test in multiple markets, pick mature markets like BTC-USDT or ETH-USDT and also volatile markets like DOGE-USDT or SHIB-USDT